In [1]:
import pandas as pd
import numpy as np
import os
import json
import random

In [ ]:
import torch
import random
import numpy as np
from itertools import compress
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
import json

def generate_scaffold(smiles, include_chirality=False):
    # mol = Chem.MolFromSmiles(smiles)
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(smiles=smiles, includeChirality=include_chirality)
    return scaffold

def scaffold_split(dataset_df, frac_train=0.8, frac_valid=0.1, frac_test=0.1, return_indices=False, seed=2024):
    """
    Split dataset into train, validation and test based on scaffold.
    """
    # Set seed
    random.seed(seed)
    np.random.seed(seed)

    # Get list of smiles
    smiles_list = dataset_df.smiles.tolist()

    rng = np.random.RandomState(seed)
   
    # Generate scaffolds
    scaffolds = defaultdict(list)
    for ind, smiles in enumerate(smiles_list):
        scaffold = generate_scaffold(smiles, include_chirality=False)
        scaffolds[scaffold].append(ind)
    
    # # 把scaffolds保存到字典中
    # with open('/home/liujin/project/BioAct-Het-main/Data/ADR_multitask_dataset_scaffold_split/scaffolds.json', 'w') as f:
    #     json.dump(scaffolds, f)

    # Shuffle scaffolds
    scaffold_sets = list(scaffolds.values())
    random.shuffle(scaffold_sets)

    # scaffold_sets = rng.permutation(list(scaffolds.values()))

    n_total_valid = int(np.floor(frac_valid * len(dataset_df)))
    n_total_test = int(np.floor(frac_test * len(dataset_df)))

    train_idx = []
    valid_idx = []
    test_idx = []

    for scaffold_set in scaffold_sets:
        if len(valid_idx) + len(scaffold_set) <= n_total_valid:
            valid_idx.extend(scaffold_set)
        elif len(test_idx) + len(scaffold_set) <= n_total_test:
            test_idx.extend(scaffold_set)
        else:
            train_idx.extend(scaffold_set)

    train_dataset_df = dataset_df.iloc[train_idx]
    valid_dataset_df = dataset_df.iloc[valid_idx]
    test_dataset_df = dataset_df.iloc[test_idx]

    if return_indices:
        return train_dataset_df, valid_dataset_df, test_dataset_df, train_idx, valid_idx, test_idx
    else:
        return train_dataset_df, valid_dataset_df, test_dataset_df

    
    

def random_split(dataset_df, frac_train=0.8, frac_valid=0.1, frac_test=0.1, return_indices=False, seed=2024):
    """
    Split dataset into train, validation and test randomly.
    """
    # Set seed
    random.seed(seed)
    np.random.seed(seed)

    # Get list of indices
    indices = dataset_df.index.tolist()
    random.shuffle(indices)

    # Calculate split sizes
    num_total = len(indices)
    num_train = int(num_total * frac_train)
    num_valid = int(num_total * frac_valid)
    num_test = num_total - num_train - num_valid
        

    # Split indices
    train_indices = indices[:num_train]
    valid_indices = indices[num_train:num_train + num_valid]
    test_indices = indices[num_train + num_valid:]

    # Create new datasets
    train_dataset_df = dataset_df.iloc[train_indices]
    valid_dataset_df = dataset_df.iloc[valid_indices]
    test_dataset_df = dataset_df.iloc[test_indices]

    if return_indices:
        return train_dataset_df, valid_dataset_df, test_dataset_df, train_indices, valid_indices, test_indices
    else:
        return train_dataset_df, valid_dataset_df, test_dataset_df

In [3]:
multitask_df = pd.read_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset/mutiltask_adr_data.csv")
print(multitask_df.shape)

(9202, 19)


In [4]:

def data_split(type,all_feature_data,train_smiles_list,valid_smiles_list,test_smiles_list,save_path):
    save_path_type = os.path.join(save_path,type)
    if not os.path.exists(save_path_type):
        os.makedirs(save_path_type)
    train_df_X = all_feature_data[all_feature_data["smiles"].isin(train_smiles_list)]
    valid_df_X = all_feature_data[all_feature_data["smiles"].isin(valid_smiles_list)]
    test_df_X = all_feature_data[all_feature_data["smiles"].isin(test_smiles_list)]
    train_df_X.to_csv(os.path.join(save_path_type,"train_X.csv"),index=False)
    valid_df_X.to_csv(os.path.join(save_path_type,"valid_X.csv"),index=False)
    test_df_X.to_csv(os.path.join(save_path_type,"test_X.csv"),index=False)

#### 1. Random split

In [5]:
train_dataset_df, valid_dataset_df, test_dataset_df = random_split(multitask_df, frac_train=0.8, frac_valid=0.1, frac_test=0.1, return_indices=False, seed=2024)
print(train_dataset_df.shape, valid_dataset_df.shape, test_dataset_df.shape)
train_dataset_df.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_random_split/train_y.csv", index=False)
valid_dataset_df.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_random_split/valid_y.csv", index=False)
test_dataset_df.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_random_split/test_y.csv", index=False)

(7361, 19) (920, 19) (921, 19)


In [6]:
train_smiles_list = train_dataset_df.smiles.tolist()
valid_smiles_list = valid_dataset_df.smiles.tolist()
test_smiles_list = test_dataset_df.smiles.tolist()

In [7]:
# AC50_Cmax_ECFP4
all_feature_data = pd.read_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset/single_task/all_data.csv")
columns_list = all_feature_data.columns.tolist()
train_df_X = all_feature_data[all_feature_data["smiles"].isin(train_smiles_list)]
valid_df_X = all_feature_data[all_feature_data["smiles"].isin(valid_smiles_list)]
test_df_X = all_feature_data[all_feature_data["smiles"].isin(test_smiles_list)]

print("train_df_X:",train_df_X.shape)
print("valid_df_X:",valid_df_X.shape)
print("test_df_X:",test_df_X.shape)

train_df_X.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_random_split/train_X.csv",index=False)
valid_df_X.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_random_split/valid_X.csv",index=False)
test_df_X.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_random_split/test_X.csv",index=False)

train_df_X: (7361, 391)
valid_df_X: (920, 391)
test_df_X: (921, 391)


In [8]:
test_df_X.head()

,Drug,smiles,cmax_free(uM),P35414,P24530,P30518,P35346,Q9Y271,P34998,P30874,...,P23975*Cmax,P31645*Cmax,Q01959*Cmax,P31652*Cmax,P23977*Cmax,Q9GZV3*Cmax,P28572*Cmax,P30536*Cmax,P01375*Cmax,Q99720*Cmax
6,pristimerin,COC(=O)[C@]1(C)CC[C@]2(C)CC[C@]3(C)C4=CC=C5C(=...,-3.592627,-0.849710,0.626633,-1.692090,-0.495694,0.234367,-0.484260,0.789575,...,1.247733,5.057295,1.215029,1.824945,0.573687,-7.198354,-0.650695,3.013458,1.649426,-1.892970
10,SIDER1181,Nc1ccn(C2CCC(CO)O2)c(=O)n1,0.420390,0.356353,1.355362,-0.421459,0.593037,0.657542,0.908293,0.529987,...,-0.186125,0.223921,0.035833,-0.093917,-0.177112,0.363736,-0.370552,-0.273816,0.342971,0.143286
14,SIDER227,O=C(O)CN(CCN(CC(=O)O)CC(=O)O)CCN(CC(=O)O)CC(=O)O,0.466524,-2.508975,-0.500437,-1.325006,0.298561,-0.515216,1.189656,-0.055481,...,1.189807,0.571546,0.426302,-0.420050,0.212398,0.557145,-1.608018,0.866297,0.269220,0.439386
22,rivoceranib,N#CC1(c2ccc(NC(=O)c3cccnc3NCc3ccncc3)cc2)CCCC1,-0.899579,-0.043746,0.264603,0.000905,-0.980561,-1.527472,-0.832586,0.403289,...,0.075487,0.236225,-0.023706,0.458785,0.228788,-0.647307,0.791778,-0.545677,-0.337574,0.076520
32,2'-deoxyguanosine-5'-diphosphate,Nc1nc2c(ncn2[C@H]2C[C@H](O)[C@@H](CO[P@](=O)(O...,0.662592,-0.798177,0.469325,-1.732069,0.561922,0.467137,0.362875,-0.125471,...,-1.021602,-1.470114,0.867722,-10.913857,-0.549442,0.716933,-0.285439,-0.681371,0.271115,0.831049


#### 2. Scaffold spilt

In [9]:
train_scaffold_df, valid_scaffold_df, test_scaffold_df = scaffold_split(multitask_df, frac_train=0.8, frac_valid=0.09, frac_test=0.1, return_indices=False, seed=2024)
print(train_scaffold_df.shape, valid_scaffold_df.shape, test_scaffold_df.shape)
train_scaffold_df.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_scaffold_split/train_y.csv", index=False)
valid_scaffold_df.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_scaffold_split/valid_y.csv", index=False)
test_scaffold_df.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_scaffold_split/test_y.csv", index=False)

(7454, 19) (828, 19) (920, 19)


In [10]:
train_smiles_list_scaff = train_scaffold_df.smiles.tolist()
valid_smiles_list_scaff = valid_scaffold_df.smiles.tolist()
test_smiles_list_scaff = test_scaffold_df.smiles.tolist()
# AC50_Cmax_ECFP4
all_feature_data = pd.read_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset/single_task/all_data.csv")
columns_list = all_feature_data.columns.tolist()
train_df_X = all_feature_data[all_feature_data["smiles"].isin(train_smiles_list_scaff)]
valid_df_X = all_feature_data[all_feature_data["smiles"].isin(valid_smiles_list_scaff)]
test_df_X = all_feature_data[all_feature_data["smiles"].isin(test_smiles_list_scaff)]

print("train_df_X:",train_df_X.shape)
print("valid_df_X:",valid_df_X.shape)
print("test_df_X:",test_df_X.shape)

train_df_X.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_scaffold_split/train_X.csv",index=False)
valid_df_X.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_scaffold_split/valid_X.csv",index=False)
test_df_X.to_csv("/home/datahouse1/liujin/HetSia-SafeNet/Data/ADR_multitask_dataset_scaffold_split/test_X.csv",index=False)

train_df_X: (7454, 391)
valid_df_X: (828, 391)
test_df_X: (920, 391)
